# 05: Performance Deviation Analysis & Root Cause Diagnosis

### Operational Concept:
$	ext{Residual} = P_{\text{actual}} - P_{\text{predicted}}$
- A negative residual indicates that actual generation was lower than expected (underperformance).
- Rather than assuming equipment failure, we use **Median Absolute Deviation (MAD)** calibrated on training data and **irradiance-adaptive threshold scaling**:
$$\text{Threshold}(I) = \text{BaseThreshold} \times \max(0.2, \min(1.5, I / 0.5))$$
This reflects the fact that solar measurement uncertainty is naturally proportional to solar irradiance.


In [ ]:
import sys
import json
import joblib
import pandas as pd
import numpy as np

sys.path.insert(0, '../src')
from deviation_detection import SolarDeviationDetector
from prediction import SolarPredictor

detector = joblib.load('../models/deviation_detector.joblib')
print('Fitted Deviation Thresholds (from training residuals):')
for k, v in detector.thresholds.items():
    print(f'  {k:22s}: {v}')


### 2. Test Set Deviation Severity Breakdown
Inspecting the distribution of operational events across the unseen test period:


In [ ]:
dev_summary = pd.read_csv('../reports/deviation_summary.csv')
print('Test Period Deviation Severity Counts:')
print(dev_summary['severity'].value_counts())
print('\nSeverity Percentage:')
print((dev_summary['severity'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')


### 3. Interactive Scenario Testing
Let us test 4 distinct operational situations:
1. **Healthy Midday Generation**: Full power matching expectation.
2. **Severe Inverter Trip**: High irradiance (0.85 kW/m²) but 90% power loss.
3. **Thermal Derating**: Extreme panel temperature (62°C) derating power.
4. **Nocturnal Solar Gating**: Night hours returning guaranteed 0.0 kW.


In [ ]:
predictor = SolarPredictor(
    model_path='../models/best_model.joblib',
    pipeline_path='../models/preprocessing_pipeline.joblib',
    metadata_path='../models/model_metadata.json',
)

# Scenario 1: Healthy Noon
s1 = predictor.predict_single('2020-06-15 12:00:00', 4135001, 0.85, 34.0, 50.0)
d1 = detector.analyze_single(s1['predicted_ac_power'] * 0.98, s1['predicted_ac_power'], 0.85, 34.0, 50.0, 4135001)
print('Scenario 1 (Healthy Noon):', d1['severity'])

# Scenario 2: Inverter Trip
d2 = detector.analyze_single(200.0, s1['predicted_ac_power'], 0.85, 34.0, 50.0, 4135001)
print('Scenario 2 (Inverter Trip):', d2['severity'], '| Attribution:', d2['contributing_conditions'])

# Scenario 3: Thermal Derating
d3 = detector.analyze_single(s1['predicted_ac_power'] - 1200.0, s1['predicted_ac_power'], 0.85, 38.0, 62.0, 4135001)
print('Scenario 3 (Thermal Derating):', d3['severity'], '| Attribution:', d3['contributing_conditions'])

# Scenario 4: Night Gating
s4 = predictor.predict_single('2020-06-15 01:00:00', 4135001, 0.0, 24.0, 22.0)
print('Scenario 4 (Night): Predicted Power =', s4['predicted_ac_power'], 'kW')
